In [0]:
#Para el total nacional calcule (a Junio de 2026):
#población en edad de trabajar
#población que no está en edad de trabajar
#población económicamente activa
#población económicamente inactiva
#tasa de ocupación
#tasa de desempleo

In [0]:
#población en edad de trabajar
import pandas as pd
import requests
from io import StringIO
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")
ocupados.display()


# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()
ocupados= ocupados.toPandas()



In [0]:

cgse.head()


In [0]:
cgse[['PERIODO', 'MES', 'P6040', 'FEX_C18']].head(20)
#El valor FEX_C18 es el factor que usamos para estimar la población representada por cada persona.

In [0]:
cgse['P6040'].head()

In [0]:
cgse.groupby(['PERIODO', 'MES']).size().reset_index(name='registros')

In [0]:
#resultado es la población total expandida de cgse, antes de filtrar por edad.
cgse
PT = cgse['FEX_C18'].sum()
PT

In [0]:
#población en edad para trabajar
#PET
PET=ft[ft["PET"]==1]["FEX_C18"].sum()
PET
#resultado de sumar los factores de expansión de las filas que cumplen PET == 1.
#41,2 millones de personas en edad de trabajar.


In [0]:
#FT = 1.0: hay 32.551 registros con ese valor.

#FT = NaN: hay 20.580 registros sin valor en esa variable.
En total son 53.131 registros.

ft['FT'].value_counts(dropna=False)

In [0]:
#POBLACIÓN QUE NO ESTA EN EDAD PARA TRABAJAR: Según lo leído en DANE profesor , 15 sale de la definición de la Población en Edad de Trabajar (PET) que utiliza el DANE en la GEIH
PNE = cgse[cgse['P6040'] < 15]['FEX_C18'].sum()
PNE

#Indicador: Población que no está en edad de trabajar (PNE).
#Condición: P6040 < 15.
#Factor de expansión: FEX_C18.



Resultado estimado: 11.134.618 personas.#

In [0]:
#POBLACIÓN ECONÓMICAMENTE ACTIVA
#PEA
PEA = ft[ft['FT'] == 1]['FEX_C18'].sum()
PEA
#se seleccionó los registros con FT == 1 y sume los factores de expansión FEX_C18 para estimar la población económicamente activa.

In [0]:
#Población Económicamente Inactiva (PEI)
#Primero se necesita encontrar una variable  la condición de actividad económica inactiva
#Lo puedo deducir de esta manera tengo la PEA = 26.593.429, se podría calcular la PEI restando la PEA de la PET:
PEI = ft[(ft['PET'] == 1) & (ft['P6240'].isin([3, 4, 5, 6]))]['FEX_C18'].sum()
PEI



In [0]:
#TASA OCUPADOS
O = PEA - D
TO = O/PEA*100
TO = O/PET*100
print(f'La tasa de ocupación (TO) en Colombia para junio 2026 es de {TO} %')

In [0]:
#TASA DE DESOCUPACION
TD = (PEI / PET)*100
print(f'Tasa de desocupacion:  {TD} %')

In [0]:
cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN']].duplicated().sum()

In [0]:
desempleados_sexo = desempleados.merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

In [0]:
desempleados_sexo[['P3271', 'DSI', 'FEX_C18']].head()

In [0]:
#PUNTO 2
# 1.Cantidad de mujeres desempleadas en el quindio
MDC = desempleados_sexo[
    (desempleados_sexo['P3271'] == 2) &
    (desempleados_sexo['DSI'] == 1)
]['FEX_C18'].sum()

print(f"En Colombia hay {MDC:,.0f} mujeres desempleadas")

# 2.Cantidad de hombres desempleados en el quindio
HDC = desempleados_sexo[
    (desempleados_sexo['P3271'] == 1) &
    (desempleados_sexo['DSI'] == 1)
]['FEX_C18'].sum()

print(f"En Colombia hay {HDC:,.0f} hombres desempleados")

In [0]:
#RTA=P3271 == 2: seleccionamos mujeres.

#P3271 == 1: seleccionamos hombres.

#DSI == 1: seleccionamos las personas clasificadas como desocupadas.

#FEX_C18.sum(): estime la cantidad de personas representadas por la encuesta, en lugar de contar únicamente las filas.

In [0]:
#¿cuáles son los departamentos con  mayores y menores tasas de desempleo?
print(sorted(desempleados_sexo['DPTO'].dropna().unique()))

In [0]:
departamentos = {
    5: 'Antioquia',
    8: 'Atlántico',
    11: 'Bogotá D. C.',
    13: 'Bolívar',
    15: 'Boyacá',
    17: 'Caldas',
    18: 'Caquetá',
    19: 'Cauca',
    20: 'Cesar',
    23: 'Córdoba',
    25: 'Cundinamarca',
    27: 'Chocó',
    41: 'Huila',
    44: 'La Guajira',
    47: 'Magdalena',
    50: 'Meta',
    52: 'Nariño',
    54: 'Norte de Santander',
    63: 'Quindío',
    66: 'Risaralda',
    68: 'Santander',
    70: 'Sucre',
    73: 'Tolima',
    76: 'Valle del Cauca',
    81: 'Arauca',
    85: 'Casanare',
    86: 'Putumayo',
    88: 'Archipiélago de San Andrés, Providencia y Santa Catalina',
    91: 'Amazonas',
    94: 'Guainía',
    95: 'Guaviare',
    97: 'Vaupés',
    99: 'Vichada'
}

In [0]:
#NOTA:desempleados_sexo: ME Permite contar a los hombres y mujeres desocupados por departamento.

#ft: la use para obtener la población económicamente activa (PEA) de hombres y mujeres por departamento, porque allí tenemos la variable FT.

In [0]:
ft_sexo = ft.merge(
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

In [0]:
PEA_departamento = (
    ft_sexo[ft_sexo['FT'] == 1]
    .groupby(['DPTO', 'P3271'])['FEX_C18']
    .sum()
    .reset_index(name='PEA')
)
print(PEA_departamento)
#falta unir esos resultados con la cantidad de personas desempleadas para calcular las tasas.

In [0]:
desocupados_departamento = (
    desempleados_sexo[desempleados_sexo['DSI'] == 1]
    .groupby(['DPTO', 'P3271'])['FEX_C18']
    .sum()
    .reset_index(name='Desocupados')
)

In [0]:
tasas = PEA_departamento.merge(
    desocupados_departamento,
    on=['DPTO', 'P3271'],
    how='left'
)

tasas['Desocupados'] = tasas['Desocupados'].fillna(0)

tasas['Tasa_desempleo'] = (
    tasas['Desocupados'] / tasas['PEA']
) * 100

tasas['Sexo'] = tasas['P3271'].map({
    1: 'Hombres',
    2: 'Mujeres'
})

tasas['Departamento'] = tasas['DPTO'].map(departamentos)

print(
    tasas[['Departamento', 'Sexo', 'PEA',
           'Desocupados', 'Tasa_desempleo']]
    .sort_values(['Departamento', 'Sexo'])
    .to_string(index=False)
)

In [0]:
tasas['Tasa_porcentaje'] = tasas['Tasa_desempleo'].map(
    lambda x: f'{x:.2f}%'
)

print(tasas[['Departamento', 'Sexo', 'Tasa_porcentaje']])

In [0]:
print("Mayores tasas:")
print(
    tasas.nlargest(5, 'Tasa_desempleo')
    [['Departamento', 'Sexo', 'Tasa_desempleo']]
)

print("Menores tasas:")
print(
    tasas.nsmallest(5, 'Tasa_desempleo')
    [['Departamento', 'Sexo', 'Tasa_desempleo']]
)

In [0]:
#Mayores tasas:

#Análisis de las tasas de desempleo por departamento y sexo, junio de 2026**

#Según los resultados obtenidos, Putumayo registró la tasa de desempleo más alta entre las mujeres, con un 25,02 %, mientras que San Andrés, Providencia y Santa Catalina tuvo la tasa más alta entre los hombres, con un 20,98 %. También se destacaron las mujeres de Sucre, cuya tasa fue del 20,49 %.

#En contraste, Vaupés presentó las tasas más bajas: 1,78 % para las mujeres y 3,00 % para los hombres. Asimismo, Amazonas registró una tasa femenina del 2,61 % y Cauca una tasa masculina del 3,05 %.

#En general, los resultados reflejan que el desempleo varía según el departamento y el sexo, pues algunos territorios presentan tasas más elevadas en las mujeres y otros en los hombres. No obstante, estos datos no permiten establecer por sí solos las razones de esas diferencias.

In [0]:
#PUNTO3
#Tomando en consideración únicamente los siguientes grupos de edad:  - - - - - 
#18 a 24 
#24 a 30 
#30 a 40 
#40 a 50  
#60 en adelante  
#Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor 
#tasa de desempleo ? 

In [0]:
ft_edad = pd.merge(
    ft,
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P6040']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

In [0]:

import pandas as pd

# 1. Agregar la edad a la tabla de fuerza de trabajo (FT)
ft_edad = ft.merge(
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P6040']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

# 2. Crear los grupos de edad
ft_edad['Grupo_edad'] = pd.cut(
    ft_edad['P6040'],
    bins=[18, 24, 30, 40, 50, float('inf')],
    labels=['18-23', '24-29', '30-39', '40-49', '60+'],
    right=False
)

# Excluir menores de 18 y personas de 50 a 59 años
ft_edad = ft_edad[
    (ft_edad['P6040'] >= 18) &
    ((ft_edad['P6040'] < 50) | (ft_edad['P6040'] >= 60))
]

# 3. Calcular la PEA por departamento y grupo de edad
pea_edad = (
    ft_edad[ft_edad['FT'] == 1]
    .groupby(['DPTO', 'Grupo_edad'], observed=True)['FEX_C18']
    .sum()
    .reset_index(name='PEA')
)

# 4. Agregar la edad a la tabla de personas desempleadas
desempleados_edad = desempleados.merge(
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P6040']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

desempleados_edad['Grupo_edad'] = pd.cut(
    desempleados_edad['P6040'],
    bins=[18, 24, 30, 40, 50, float('inf')],
    labels=['18-23', '24-29', '30-39', '40-49', '60+'],
    right=False
)

desempleados_edad = desempleados_edad[
    (desempleados_edad['P6040'] >= 18) &
    ((desempleados_edad['P6040'] < 50) |
     (desempleados_edad['P6040'] >= 60))
]

# 5. Calcular los desempleados por departamento y grupo de edad
desocupados_edad = (
    desempleados_edad[desempleados_edad['DSI'] == 1]
    .groupby(['DPTO', 'Grupo_edad'], observed=True)['FEX_C18']
    .sum()
    .reset_index(name='Desocupados')
)

# 6. Calcular la tasa de desempleo
tasas_edad = pea_edad.merge(
    desocupados_edad,
    on=['DPTO', 'Grupo_edad'],
    how='left'
)

tasas_edad['Desocupados'] = tasas_edad['Desocupados'].fillna(0)

tasas_edad['Tasa_desempleo'] = (
    tasas_edad['Desocupados'] / tasas_edad['PEA'] * 100
)

tasas_edad['Departamento'] = tasas_edad['DPTO'].map(departamentos)

# 7. Identificar el grupo con mayor y menor tasa por departamento
tasas_validas = tasas_edad[
    (tasas_edad['PEA'] > 0) &
    tasas_edad['Tasa_desempleo'].notna()
]

indices = tasas_validas.groupby('DPTO')['Tasa_desempleo'].agg(
    ['idxmax', 'idxmin']
)

mayor = tasas_validas.loc[indices['idxmax']].copy()
menor = tasas_validas.loc[indices['idxmin']].copy()

mayor['Resultado'] = 'Mayor tasa'
menor['Resultado'] = 'Menor tasa'

resultado = pd.concat([mayor, menor])
resultado = resultado.sort_values(['DPTO', 'Tasa_desempleo'],
                                  ascending=[True, False])

print(resultado[
    ['Departamento', 'Grupo_edad', 'Tasa_desempleo', 'Resultado']
].to_string(index=False))


In [0]:
#- En la mayoría de los departamentos, las personas de 18 a 23 años presentan las tasas de desempleo más altas.

#- Sucre registra la tasa más elevada para este grupo, con un 33,14 %, seguido de Vichada con 31,60 % y Bolívar con 30,86 %.

#- En Boyacá, Chocó, Nariño, Arauca y Putumayo, el grupo de 24 a 29 años presenta la mayor tasa de desempleo.

#- En La Guajira, la tasa más alta corresponde a las personas entre 40 y 49 años.

#- Las tasas más bajas se presentan principalmente en las personas de 60 años o más, como ocurre en Meta, Cauca, Chocó y Caldas.

#- En algunos departamentos, como Boyacá, Nariño y Valle del Cauca, otros grupos de edad registran las tasas más bajas.

#- En general, los resultados muestran que el desempleo afecta de manera diferente a cada grupo de edad según el departamento.

#- Las tasas del 0 % deben revisarse, ya que podrían reflejar la ausencia de casos de desempleo observados en determinados grupos y no necesariamente la inexistencia de desempleo.)

In [0]:
#punto 4

# 1. Incorporar edad y sexo a la fuerza de trabajo
ft_mujeres_edad = ft.merge(
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN',
          'P6040', 'P3271']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

# 2. Calcular la PEA de mujeres entre 18 y 25 años
pea_mujeres = (
    ft_mujeres_edad[
        (ft_mujeres_edad['FT'] == 1) &
        (ft_mujeres_edad['P3271'] == 2) &
        (ft_mujeres_edad['P6040'].between(18, 25))
    ]
    .groupby('DPTO')['FEX_C18']
    .sum()
    .reset_index(name='PEA_mujeres_18_25')
)

# 3. Incorporar edad y sexo a los desempleados
desempleadas_edad = desempleados.merge(
    cgse[['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN',
          'P6040', 'P3271']],
    on=['PERIODO', 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left',
    validate='many_to_one'
)

# 4. Calcular las mujeres desempleadas entre 18 y 25 años
desocupadas_mujeres = (
    desempleadas_edad[
        (desempleadas_edad['DSI'] == 1) &
        (desempleadas_edad['P3271'] == 2) &
        (desempleadas_edad['P6040'].between(18, 25))
    ]
    .groupby('DPTO')['FEX_C18']
    .sum()
    .reset_index(name='Desocupadas_mujeres_18_25')
)

# 5. Calcular la tasa de desempleo
tasas_mujeres = pea_mujeres.merge(
    desocupadas_mujeres,
    on='DPTO',
    how='left'
)

tasas_mujeres['Desocupadas_mujeres_18_25'] = (
    tasas_mujeres['Desocupadas_mujeres_18_25'].fillna(0)
)

tasas_mujeres['Tasa_desempleo'] = (
    tasas_mujeres['Desocupadas_mujeres_18_25'] /
    tasas_mujeres['PEA_mujeres_18_25'] * 100
)

tasas_mujeres['Departamento'] = (
    tasas_mujeres['DPTO'].map(departamentos)
)

# 6. Mostrar los tres departamentos con menor tasa
top_3 = (
    tasas_mujeres[
        tasas_mujeres['PEA_mujeres_18_25'] > 0
    ]
    .sort_values('Tasa_desempleo')
    .head(3)
)

print(top_3[
    ['Departamento', 'Tasa_desempleo']
].to_string(index=False))


In [0]:

# Revisar mujeres de 18 a 25 años en Vaupés (DPTO = 97)
#Significa que, en los datos analizados, no se registraron mujeres desempleadas de ese grupo de edad en Vaupés. perono  quiere decir necesariamente que en la realidad ninguna mujer joven esté desempleada, sino que la estimación de esta muestra dio cero.

vaupes = desempleadas_edad[
    (desempleadas_edad['DPTO'] == 97) &
    (desempleadas_edad['P3271'] == 2) &
    (desempleadas_edad['P6040'].between(18, 25))
]

# Mujeres desempleadas registradas
vaupes_desocupadas = vaupes[vaupes['DSI'] == 1]

print("Mujeres desempleadas de 18 a 25 años en Vaupés:")
print("Cantidad de registros:", len(vaupes_desocupadas))
print("Población estimada:", vaupes_desocupadas['FEX_C18'].sum())

# Revisar la PEA de mujeres de 18 a 25 años
vaupes_pea = ft_mujeres_edad[
    (ft_mujeres_edad['DPTO'] == 97) &
    (ft_mujeres_edad['P3271'] == 2) &
    (ft_mujeres_edad['P6040'].between(18, 25)) &
    (ft_mujeres_edad['FT'] == 1)
]

print("\nMujeres en la PEA:")
print("Cantidad de registros:", len(vaupes_pea))
print("Población estimada:", vaupes_pea['FEX_C18'].sum())


In [0]:

#PUNTO 5
#Calcule el promedio de ingresos laborales (variable INGLABO) de las personas 
#residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle 
#del Cauca.
# Promedio de ingresos laborales por departamento

departamentos_seleccionados = {
    66: 'Risaralda',
    17: 'Caldas',
    63: 'Quindío',
    5: 'Antioquia',
    76: 'Valle del Cauca'
}

for codigo, nombre in departamentos_seleccionados.items():
    promedio = ocupados.loc[
        ocupados['DPTO'] == codigo, 'INGLABO'
    ].mean()

    print(f"{nombre}: ${promedio:,.2f}")


In [0]:

#PUNTO 5
# Promedio de ingresos laborales (INGLABO)
# Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca

promedio_INGLABO = ocupados[
    ocupados['DPTO'].isin([66, 17, 63, 5, 76])
]['INGLABO'].mean()

print("Promedio de ingresos laborales:", promedio_INGLABO)

#Antioquia registra el promedio de ingresos laborales más alto, con $2.754.449,60 COP.

#Valle del Cauca presenta el promedio más bajo, con $2.219.017,58 COP.

#Caldas alcanza un promedio de $2.427.893,31 COP, mientras que Quindío registra $2.289.058,93 COP.
#Risaralda presenta un promedio de $2.256.411,03 COP.

#el ingreso laboral promedio de las personas ocupadas de Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca es de aproximadamente $2.417.393 COP.
